# Capa Bronze - Catálogo de Empleados

In [1]:
from spark_session import *
from pyspark.sql.functions import col, date_format
from pyspark.sql.types import DecimalType

## SparkSession

In [2]:
spark = create_spark_session(
    "retail-dlh-bronze",
    configs=[
        configure_delta_extensions,
        configure_delta_catalog,
        configure_s3
    ]
)

## Extracción

In [3]:
# - Obtiene dataframe de la capa landing
df_employees = spark.table("landing.employees")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Transformación

In [4]:
# - Genera DATE_ENTERED_ID a partir DATE_ENTERED para usar como columna de partición
df_employees = df_employees \
    .withColumn("DATE_ENTERED_ID", date_format(col("DATE_ENTERED"),"yyyyMMdd") \
    .cast(DecimalType(8)))

## Carga

In [5]:
# - Reemplaza la información por particion en la capa bronze
df_employees.write.format("delta") \
      .option("partitionOverwriteMode", "dynamic") \
      .mode("overwrite") \
      .saveAsTable("bronze.employee")

## Vacuum

In [6]:
# - Actualiza la configuración para que pueda establecer un período de retención cero.
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

#- Ejecutemos el comando de VACUUM 
#spark.sql(f"VACUUM '{vPathBron}EMPLOYEE.delta' RETAIN 0 HOURS").show(truncate=False)

## Cierre de SparkSession

In [7]:
spark.stop()